# 03 — Wavelength Calibration

**Purpose:** Identify known spectral lines from a lamp image, record their pixel positions 
per order, and verify the resulting polynomial wavelength solution.

**When to run:** Only when recalibrating after an optical realignment or detector swap. 
For routine sessions, use the existing `Th_wavelength_CMOS_20240305.txt`.

**Inputs:**
- A ThAr, Ne, Hg, or H₂ lamp `.sif` image
- An existing pattern file (from notebook 02)

**Output:** `Th_wavelength_CMOS_NEWDATE.txt` — tab-separated table of identified lines.

**Based on:** `examples/obsolete/4.3_Wavelength_Calibration_LHD.ipynb`

---
### Wavelength table format
```
#order  from    to      center      wavelength  band
0       0632    0657    0644.5921   794.81764   ArI
```
- `order`: diffraction order index (0-based)
- `from`, `to`: pixel range containing the line
- `center`: sub-pixel centroid (from Gaussian fit)
- `wavelength`: known wavelength in nm (from NIST)
- `band`: species label (ArI, NeI, ThI, HgI, H2, ...)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lmfit.models import GaussianModel
import plotly.graph_objects as go
from ipywidgets import interact, widgets
from IPython.display import display

from echelle_spectra.tools.echelle import Calibrations, EchelleImage

import echelle_spectra
CALIB_DIR = echelle_spectra._config['base_path'] / 'resources/calibration_files'

%matplotlib inline

## Configuration

In [ ]:
# Calibration files — use the latest pattern, but the wavelength file will be rebuilt
files_cmos = {
    "orders": "pattern_CMOS_20240305.txt",
    "wavelength": "Th_wavelength_CMOS_20240305.txt",
    "sphr": "sphere_cmos_20240305.sif",
    "bkgr": "sphere_cmos_20240305_bkg.sif",
    "integral": "integrating_sphere.txt",
}

# Path to the lamp image you want to calibrate with
LAMP_IMAGE_PATH = str(CALIB_DIR / "ThAr-0.3s-x3_20240305.sif")

# Where to write the new wavelength table
OUTPUT_WAVELENGTH_FILE = "Th_wavelength_CMOS_NEWDATE.txt"  # edit NEWDATE

## Load Calibrations and Lamp Image

In [ ]:
cb = Calibrations(folder=str(CALIB_DIR), filenames=files_cmos)
cb.start()
cb.sphr.calculate_order_spectra()
cb.bkgr.calculate_order_spectra()
sphr = cb.sphr

# Estimate background per order from background image
subtract = {nord: cb.bkgr.order_spectra[0, nord] for nord in range(cb.bkgr.order_spectra.shape[1])}

# Load lamp image
th = EchelleImage(LAMP_IMAGE_PATH, clbr=cb)
th.calculate_order_spectra()
th.correct_order_shapes()
print(f"Lamp image shape: {th.order_spectra.shape}")

## Browse Orders Interactively

Use this widget to find candidate lines in each order. Note the approximate pixel position of each line you want to identify.

In [ ]:
x = np.arange(cb.DIMW)
n_orders = th.order_spectra.shape[1]

initial_nord = 3
y0 = (th.order_spectra[0, initial_nord] - subtract[initial_nord]) / sphr.order_spectra[0, initial_nord]
fig_widget = go.FigureWidget(data=[go.Scatter(x=x, y=y0, name=str(initial_nord))])
fig_widget.update_layout(template='plotly_white', title='Lamp spectrum by order')

def update_order(nord):
    y = (th.order_spectra[0, nord] - subtract[nord]) / sphr.order_spectra[0, nord]
    fig_widget.data[0].y = y
    fig_widget.data[0].name = str(nord)

interact(update_order, nord=widgets.Dropdown(options=list(range(n_orders)), value=initial_nord, description='order'))
display(fig_widget)

## Fit a Single Line

Use `fit_line()` to measure the sub-pixel center of a chosen peak. 
Then record it in the wavelength table below.

In [ ]:
def fit_line(nord, x0, d, lamp_image, sphr, subtract, cb):
    """Fit a Gaussian to a spectral line and return its centroid pixel.
    
    Parameters
    ----------
    nord : int
        Order index.
    x0 : int
        Approximate pixel position of the line center.
    d : int
        Half-width of the fitting window in pixels.
    """
    x_all = np.arange(cb.DIMW)
    y_all = (lamp_image.order_spectra[0, nord] - subtract[nord]) / sphr.order_spectra[0, nord]

    xs = x_all[x0 - d: x0 + d]
    ys = y_all[x0 - d: x0 + d]
    ys = ys - ys.min()

    gauss = GaussianModel()
    pars  = gauss.guess(ys, x=xs)
    out   = gauss.fit(ys, pars, x=xs)
    center = out.params['center'].value

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(xs, ys, 'o', label='data')
    xfine = np.linspace(xs[0], xs[-1], 300)
    ax.plot(xfine, gauss.eval(out.params, x=xfine), label='fit')
    ax.axvline(center, color='r', linestyle='--', label=f'center={center:.4f}')
    ax.set_title(f'Order {nord}, x0={x0}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Centroid: {center:.4f} px")
    return center

In [ ]:
# Example: fit a line in order 8 near pixel 454
# center = fit_line(nord=8, x0=454, d=10, lamp_image=th, sphr=sphr, subtract=subtract, cb=cb)

## Build Wavelength Table

Add a row for each identified line. Look up wavelengths from [NIST Atomic Spectra Database](https://www.nist.gov/pml/atomic-spectra-database).

Format: `(order, pixel_from, pixel_to, centroid, wavelength_nm, species)`

In [ ]:
# Build the wavelength table manually.
# Each tuple: (order, from_px, to_px, center_px, wavelength_nm, species)
#
# Start from the existing calibration and add/correct lines:
existing_wl_file = CALIB_DIR / "Th_wavelength_CMOS_20240305.txt"
names = ['ord', 'from', 'to', 'center', 'wavelength', 'band']
wcal = pd.read_csv(existing_wl_file, sep='\t', comment='#', names=names)
print(f"Loaded {len(wcal)} lines from existing calibration file.")
wcal.head(10)

## Verify: Plot Wavelength Solution Per Order

In [ ]:
# Polynomial degree: 1 if ≤2 lines, 2 if ≥3 lines
deg = lambda n: 1 if n < 3 else 2

fig, axes = plt.subplots(5, 6, figsize=(18, 12), sharex=False)
axes = axes.flatten()

for j, nord in enumerate(range(29)):
    ax = axes[j]
    subset = wcal[wcal['ord'] == nord]
    if len(subset) == 0:
        ax.set_title(f'Order {nord} — NO DATA')
        continue
    p = subset['center'].values
    w = subset['wavelength'].values
    poly = np.poly1d(np.polyfit(p, w, deg(len(p))))
    xfit = np.linspace(p.min(), p.max(), 200)
    ax.plot(p, w, 'o', markersize=4)
    ax.plot(xfit, poly(xfit), '-')
    residuals = w - poly(p)
    ax.set_title(f'Order {nord} (n={len(p)}, rms={residuals.std():.3f} nm)', fontsize=7)

for j in range(29, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Wavelength calibration — fit residuals', fontsize=11)
plt.tight_layout()
plt.show()

## Verify: Browse Calibrated Order Spectra

In [ ]:
# Apply calibration and browse orders vs wavelength
th.calibrate()

x_px = np.arange(cb.DIMW)
fig_cal = go.FigureWidget(data=[go.Scatter(
    x=th.clbr.order_wavel[initial_nord],
    y=th.order_spectra[0][initial_nord],
    name=str(initial_nord)
)])
fig_cal.update_layout(template='plotly_white', xaxis_title='Wavelength (nm)', title='Calibrated lamp spectrum')

def update_cal(nord):
    fig_cal.data[0].x = th.clbr.order_wavel[nord]
    fig_cal.data[0].y = th.order_spectra[0][nord]
    fig_cal.data[0].name = str(nord)

interact(update_cal, nord=widgets.Dropdown(options=list(range(n_orders)), value=initial_nord, description='order'))
display(fig_cal)

## Save Wavelength Table

In [ ]:
# When satisfied, copy the modified wcal dataframe to the calibration directory.
SAVE = False  # set to True to write

if SAVE:
    out_path = CALIB_DIR / OUTPUT_WAVELENGTH_FILE
    header_line = "#order\tfrom\tto\tcenter\twavelength\tband\n"
    with open(out_path, 'w') as f:
        f.write(header_line)
        wcal.to_csv(f, sep='\t', index=False, header=False)
    print(f"Saved: {out_path}")
else:
    print("SAVE=False — not writing file.")